# AI CV Personality Analyzer
## 02 - Data Preprocessing

### Project Overview

This notebook prepares the raw personality dataset for Natural Language
Processing (NLP) and Machine Learning.

The raw dataset contains CV/essay text and binary Big Five personality
labels:

- **O** — Openness
- **C** — Conscientiousness
- **E** — Extraversion
- **A** — Agreeableness
- **N** — Neuroticism

### Objectives

In this notebook, we will:

1. Load the raw train, validation, and test datasets.
2. Remove unnecessary index information.
3. Convert personality labels from strings to integers.
4. Validate the target labels.
5. Clean and normalize the text.
6. Handle empty text records.
7. Check duplicate records.
8. Verify the processed datasets.
9. Save the processed datasets for feature engineering.

### Important Data Leakage Rule

The predefined training, validation, and test splits will be preserved.

We will **not combine the three datasets** during preprocessing.

The raw data will remain unchanged.

## 1. Import Required Libraries

The following libraries are used for:

- Data manipulation
- Text processing
- Regular expressions
- File and directory management

In [1]:
# Import pandas for DataFrame operations
import pandas as pd

# Import NumPy for numerical operations
import numpy as np

# Import regular expressions for text cleaning
import re

# Import os for directory and file operations
import os

# Display all columns when viewing DataFrames
pd.set_option("display.max_columns", None)

# Display wider DataFrames in the notebook
pd.set_option("display.width", 120)

print("Libraries imported successfully.")

Libraries imported successfully.


## 2. Define Dataset Paths

The raw datasets are stored in:

`data/raw/`

The processed datasets will be saved to:

`data/processed/`

The original raw files will not be modified.

In [2]:
# Define paths to the raw datasets
TRAIN_PATH = "../data/raw/train.parquet"
VALIDATION_PATH = "../data/raw/validation.parquet"
TEST_PATH = "../data/raw/test.parquet"

# Define the directory where processed datasets will be stored
PROCESSED_DIR = "../data/processed"

# Create the processed directory if it does not already exist
os.makedirs(PROCESSED_DIR, exist_ok=True)

print("Raw dataset paths configured.")
print("Processed data directory:", PROCESSED_DIR)

Raw dataset paths configured.
Processed data directory: ../data/processed


## 3. Load Raw Datasets

Load the three predefined dataset splits.

We keep the original train, validation, and test separation to avoid
data leakage during later Machine Learning stages.

In [3]:
# Load the training dataset
train_df = pd.read_parquet(TRAIN_PATH)

# Load the validation dataset
validation_df = pd.read_parquet(VALIDATION_PATH)

# Load the test dataset
test_df = pd.read_parquet(TEST_PATH)

# Display the dataset sizes
print("Datasets loaded successfully.")
print("-" * 40)
print(f"Training Set   : {train_df.shape}")
print(f"Validation Set : {validation_df.shape}")
print(f"Test Set       : {test_df.shape}")

Datasets loaded successfully.
----------------------------------------
Training Set   : (1578, 8)
Validation Set : (395, 8)
Test Set       : (494, 8)


## 4. Initial Data Validation

Before making any changes, verify that all expected columns are
available in each dataset.

Expected columns:

- O
- C
- E
- A
- N
- ptype
- text
- __index_level_0__

In [4]:
# Display columns in each dataset

print("Training columns:")
print(train_df.columns.tolist())

print("\nValidation columns:")
print(validation_df.columns.tolist())

print("\nTest columns:")
print(test_df.columns.tolist())

Training columns:
['O', 'C', 'E', 'A', 'N', 'ptype', 'text', '__index_level_0__']

Validation columns:
['O', 'C', 'E', 'A', 'N', 'ptype', 'text', '__index_level_0__']

Test columns:
['O', 'C', 'E', 'A', 'N', 'ptype', 'text', '__index_level_0__']


## 5. Remove Unnecessary Index Column

The raw dataset contains:

`__index_level_0__`

This column represents the original row index from the source dataset.
It does not contain useful information for predicting personality.

We will remove it from all three datasets.

The original Parquet files remain unchanged.

In [5]:
# Name of the unnecessary source index column
INDEX_COLUMN = "__index_level_0__"

# Remove the column only if it exists
for df in [train_df, validation_df, test_df]:

    if INDEX_COLUMN in df.columns:
        df.drop(columns=[INDEX_COLUMN], inplace=True)

# Confirm the column has been removed
print("Index column removed.")

print("\nRemaining columns:")
print(train_df.columns.tolist())

Index column removed.

Remaining columns:
['O', 'C', 'E', 'A', 'N', 'ptype', 'text']


## 6. Define Personality Target Columns

The Big Five personality traits are the prediction targets:

- **O** — Openness
- **C** — Conscientiousness
- **E** — Extraversion
- **A** — Agreeableness
- **N** — Neuroticism

Each target is a binary classification problem with labels `0` and `1`.

In [6]:
# Define the Big Five personality target columns
TARGET_COLUMNS = ["O", "C", "E", "A", "N"]

# Verify that all target columns exist
missing_targets = [
    column for column in TARGET_COLUMNS
    if column not in train_df.columns
]

# Stop the pipeline if any target is missing
if missing_targets:
    raise ValueError(
        f"Missing personality target columns: {missing_targets}"
    )

print("All Big Five target columns are available.")
print(TARGET_COLUMNS)

All Big Five target columns are available.
['O', 'C', 'E', 'A', 'N']


## 7. Convert Personality Labels to Numeric Values

The raw dataset stores the Big Five labels as strings:

- `'0'`
- `'1'`

Machine Learning algorithms require numerical target values.

Therefore, we convert:

`'0' → 0`

`'1' → 1`

We apply the same conversion independently to the train,
validation, and test datasets.

In [7]:
# Convert each Big Five target from string labels to integers

for df in [train_df, validation_df, test_df]:

    for column in TARGET_COLUMNS:

        # Convert values to numeric
        df[column] = pd.to_numeric(
            df[column],
            errors="raise"
        ).astype(int)

print("Personality labels converted to integers.")

Personality labels converted to integers.


## 8. Validate Personality Labels

After conversion, each personality target should contain only:

- `0`
- `1`

Any other value indicates a data-quality problem.

In [8]:
# Validate the target values in all three datasets

for dataset_name, df in {
    "Train": train_df,
    "Validation": validation_df,
    "Test": test_df
}.items():

    print("\n" + "=" * 50)
    print(dataset_name)
    print("=" * 50)

    for column in TARGET_COLUMNS:

        unique_values = sorted(df[column].unique())

        print(f"{column}: {unique_values}")

        # Ensure only binary labels are present
        if not set(unique_values).issubset({0, 1}):
            raise ValueError(
                f"Unexpected labels found in {dataset_name} - {column}"
            )


Train
O: [0, 1]
C: [0, 1]
E: [0, 1]
A: [0, 1]
N: [0, 1]

Validation
O: [0, 1]
C: [0, 1]
E: [0, 1]
A: [0, 1]
N: [0, 1]

Test
O: [0, 1]
C: [0, 1]
E: [0, 1]
A: [0, 1]
N: [0, 1]


## 9. Analyze the Text Column

The `text` column contains the written content that will eventually
be used by the NLP model.

Before cleaning the text, we inspect:

- Missing values
- Empty strings
- Whitespace-only records
- Data type

In [9]:
# Check the text column before preprocessing

for dataset_name, df in {
    "Train": train_df,
    "Validation": validation_df,
    "Test": test_df
}.items():

    print("\n" + "=" * 50)
    print(dataset_name)
    print("=" * 50)

    print("Data type:", df["text"].dtype)
    print("Missing values:", df["text"].isnull().sum())

    # Convert temporarily to strings for empty-text checking
    text_values = df["text"].fillna("").astype(str)

    print(
        "Empty/whitespace-only records:",
        text_values.str.strip().eq("").sum()
    )


Train
Data type: object
Missing values: 0
Empty/whitespace-only records: 0

Validation
Data type: object
Missing values: 0
Empty/whitespace-only records: 0

Test
Data type: object
Missing values: 0
Empty/whitespace-only records: 0


## 10. Handle Missing Text

Text is the primary input for the personality prediction system.

Records without meaningful text cannot provide useful information
for NLP-based personality prediction.

We will:

1. Convert missing text to an empty string.
2. Remove records where the text contains only whitespace.

This operation is applied independently to each dataset split.

In [10]:
# Replace missing text values with empty strings
for df in [train_df, validation_df, test_df]:
    df["text"] = df["text"].fillna("").astype(str)

# Remove records with empty or whitespace-only text
train_df = train_df[
    train_df["text"].str.strip().ne("")
].copy()

validation_df = validation_df[
    validation_df["text"].str.strip().ne("")
].copy()

test_df = test_df[
    test_df["text"].str.strip().ne("")
].copy()

# Reset indexes after removing records
train_df.reset_index(drop=True, inplace=True)
validation_df.reset_index(drop=True, inplace=True)
test_df.reset_index(drop=True, inplace=True)

print("Empty text records removed.")

Empty text records removed.
